# Phase 4 — Partitioning Analysis (CircuitNet-N28)

Runs and compares the four Phase 4 partitioning strategies. Design decisions come
directly from Section C of `feature_analysis_N28.ipynb`:

- `design_name` = **PRIMARY** (client axis)
- `clock_ns`, `utilization` = **SECONDARY** (persona, imposed by the partitioner)
- `macro_count`, `macro_placement`, `power_mesh` = **flow** (only used to stratify P0)

Strategies implemented:

| id | strategy | class |
| -- | -------- | ----- |
| P0 | IID baseline (round-robin within flow-stratified cells) | `IIDPartitioner` |
| P1 | Hierarchical deterministic: `design_name × clock_bin` | `HierarchicalPersonaPartitioner` |
| P2 | Metadata-Dirichlet on `design_name` (α sweep) | `FeatureDirichletPartitioner` |
| P3 | Data-driven k-prototypes on weighted mixed-type features (validation of P1) | `KPrototypesPartitioner` |

All strategies emit the same number of clients (`N_CLIENTS = 12`) so the JS-divergence,
size-imbalance and label-composition scores are directly comparable. This matches the
natural P1 grid: 6 design variants × 2 quantile bins of `clock_ns`.

P3 uses the weight schema you specified (with `design_name` replacing `family`):

| feature | kind | weight |
| ------- | ---- | ------ |
| design_name | nominal | 2.00 |
| size_class | ordinal | 1.00 |
| log_frequency | interval | 1.50 |
| utilization | interval | 1.50 |
| aspect_ratio | interval | 1.00 |
| power_mesh | nominal | 0.75 |
| macro_placement | nominal | 0.75 |
| filler | nominal | 0.50 |

On N28 `size_class` collapses to a single value and `aspect_ratio` is not part of the
design space; the partitioner drops them automatically (`dropped_features_`), so the
same spec transfers unchanged to N14.

# Imports

In [ ]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as mcm
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.figsize': (14, 5),
    'axes.titlesize': 11,
    'axes.grid': True,
    'grid.color': 'white',
    'grid.linewidth': 0.8,
    'axes.facecolor': '#f5f5f5',
    'figure.facecolor': 'white',
})

sys.path.insert(0, os.path.abspath('.'))
from partitioning import (
    IIDPartitioner,
    HierarchicalPersonaPartitioner,
    FeatureDirichletPartitioner,
    KPrototypesPartitioner,
)

FEATURE_DIR = '../routability_ir_drop_prediction/training_set_N28/DRC/feature'
OUT_DIR = './partition_outputs'
os.makedirs(OUT_DIR, exist_ok=True)
print('Imports OK.')

# Load metadata (same parser as `feature_analysis_N28.ipynb`)

In [ ]:
def parse_sample_name(filename: str) -> dict:
    basename = filename.replace('.npy', '')
    parts = basename.split('-')
    if parts[0].isdigit():
        parts = parts[1:]
    if len(parts) < 7:
        raise ValueError(f'Cannot parse (too few tokens): {filename}')
    for expected, token in zip(['c', 'u', 'm', 'p', 'f'], parts[-5:]):
        if not token.startswith(expected):
            raise ValueError(
                f'Cannot parse {filename}: expected prefix "{expected}", got "{token}"'
            )
    clock_str, util_str, macro_placement_raw, power_mesh_raw, filler_raw = parts[-5:]
    macro_count = parts[-6]
    design_name = '-'.join(parts[:-6])
    if not design_name:
        raise ValueError(f'Cannot parse {filename}: empty design name')
    return {
        'design_name':      design_name,
        'macro_count':      macro_count,
        'clock_ns':         float(clock_str[1:]),
        'utilization':      float(util_str[1:]),
        'macro_placement':  macro_placement_raw[1:],
        'power_mesh':       power_mesh_raw[1:],
        'filler_insertion': filler_raw[1:],
        'filename':         filename,
    }


files = sorted(f for f in os.listdir(FEATURE_DIR) if f.endswith('.npy'))
records = []
for fname in files:
    try:
        records.append(parse_sample_name(fname))
    except Exception as e:
        print(f'  Skip {fname}: {e}')
df_meta = pd.DataFrame(records)
df_meta.loc[df_meta['macro_placement'].isin(['1','2','3','4']) == False, 'macro_placement'] = '1'
df_meta['macro_placement'] = df_meta['macro_placement'].astype(str)
df_meta['power_mesh']      = df_meta['power_mesh'].astype(str)
df_meta['filler_insertion']= df_meta['filler_insertion'].astype(str)
print(f'Samples: {len(df_meta)}   Designs: {df_meta["design_name"].nunique()}')
print(df_meta.head())

# Derived features for P3 (`size_class`, `log_frequency`, `filler`, `aspect_ratio`)

The P3 spec is dataset-agnostic (works for N28 or N14). We derive:

- `size_class` — small / medium / large, keyed directly on `design_name`. All N28
  designs are `small` RISC-V variants, so this column is constant on N28 and gets
  dropped by `KPrototypesPartitioner` automatically. On N14 it separates {small
  RISC-V, OpenC910/Vortex-small, Vortex-large/NVDLA}.
- `log_frequency` — `log10(1000 / clock_ns)` gives the frequency in MHz on a log
  scale, which is the natural spacing for the {50, 200, 500} MHz sweep in N28.
- `filler` — human-readable `after_placement` / `after_routing` from the {0, 1} raw code.
- `aspect_ratio` — constant 1.0 on N28 (no aspect ratio axis in this dataset); dropped
  automatically by the partitioner. Present as a real axis on N14.

In [ ]:
SIZE_MAP = {
    'zero-riscy-a': 'small', 'zero-riscy-b': 'small',
    'RISCY-a':      'small', 'RISCY-b':      'small',
    'RISCY-FPU-a':  'small', 'RISCY-FPU-b':  'small',
    'OpenC910-1':   'medium', 'Vortex-small': 'medium',
    'Vortex-large': 'large',  'NVDLA-small':  'large', 'NVDLA-large': 'large',
}
FILLER_MAP = {'0': 'after_routing', '1': 'after_placement'}

df_meta['size_class']    = df_meta['design_name'].map(SIZE_MAP).fillna('small')
df_meta['log_frequency'] = np.log10(1000.0 / df_meta['clock_ns'].astype(float))
df_meta['filler']        = df_meta['filler_insertion'].map(FILLER_MAP).fillna(df_meta['filler_insertion'])
df_meta['aspect_ratio']  = 1.0

print('Derived column summary:')
for col in ['size_class','log_frequency','filler','aspect_ratio']:
    vc = df_meta[col].value_counts()
    if len(vc) <= 10:
        print(f'  {col:15s} -> {dict(vc)}')
    else:
        print(f'  {col:15s} -> {len(vc)} unique values')
print()
print(df_meta[['design_name','size_class','clock_ns','log_frequency','filler']].head())

# Scoring helpers

Every partitioning is scored with the same three descriptors:

1. **Size distribution** — sample count per client, and the min/median/max
   imbalance ratio.
2. **Mean JS divergence to the global distribution** — for every factor of
   interest (`design_name`, `clock_ns`, `utilization`, `macro_placement`,
   `power_mesh`, `filler_insertion`), we compute the JSD of the client's
   marginal against the global marginal, then average across clients.
   Higher = the partitioner concentrates that factor into specific clients
   (which is what we *want* for the design axis, but *not* for the flow axis).
3. **Composition stack-plots** — the per-client marginal of each factor,
   for visual inspection.

In [ ]:
FACTOR_COLS = [
    'design_name', 'clock_ns', 'utilization',
    'macro_placement', 'power_mesh', 'filler_insertion',
]

global_props = {c: df_meta[c].astype(str).value_counts(normalize=True) for c in FACTOR_COLS}


def per_client_composition(parts, col):
    cols = []
    for i, p in enumerate(parts):
        vc = p[col].astype(str).value_counts(normalize=True)
        vc.name = f'c{i}'
        cols.append(vc)
    return pd.concat(cols, axis=1).fillna(0.0).sort_index()


def js_divergence_to_global(parts, col, global_dist):
    all_levels = sorted(set(global_dist.index) | set().union(
        *[set(p[col].astype(str).unique()) for p in parts]
    ))
    g = global_dist.reindex(all_levels).fillna(0.0).to_numpy()
    g = g / max(g.sum(), 1e-12)
    out = []
    for p in parts:
        vc = p[col].astype(str).value_counts(normalize=True)
        v = vc.reindex(all_levels).fillna(0.0).to_numpy()
        if v.sum() == 0:
            out.append(np.nan)
        else:
            v = v / v.sum()
            # scipy.jensenshannon returns sqrt(JSD); square to recover JSD.
            out.append(float(jensenshannon(v, g, base=2) ** 2))
    return out


def summarize(parts, name, factor_cols=FACTOR_COLS):
    sizes = [len(p) for p in parts]
    print(f'\n=== {name} ===')
    print(f'  clients            : {len(parts)}')
    print(f'  size min/median/max: {min(sizes)} / {int(np.median(sizes))} / {max(sizes)}')
    imb = max(sizes) / max(min(sizes), 1)
    print(f'  size imbalance     : max/min = {imb:.2f}x')
    js = {}
    for col in factor_cols:
        js[col] = float(np.nanmean(js_divergence_to_global(parts, col, global_props[col])))
    for col, v in js.items():
        print(f'  mean JS({col:20s}) = {v:.3f}')
    return {'name': name, 'sizes': sizes, 'imbalance': imb, 'js_mean': js}


def plot_partition_composition(parts, name, cols=('design_name','clock_ns','utilization')):
    fig, axes = plt.subplots(1, len(cols), figsize=(6 * len(cols), 5))
    if len(cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, cols):
        comp = per_client_composition(parts, col)
        levels = comp.index.tolist()
        n_clients = comp.shape[1]
        cmap = mcm.get_cmap('tab20', max(len(levels), 3))
        bottom = np.zeros(n_clients)
        for j, lvl in enumerate(levels):
            h = comp.iloc[j].to_numpy()
            ax.bar(range(n_clients), h, bottom=bottom,
                   color=cmap(j % cmap.N), label=str(lvl),
                   alpha=0.9, edgecolor='white', linewidth=0.5)
            bottom += h
        ax.set_xticks(range(n_clients))
        ax.set_xticklabels([f'c{i}' for i in range(n_clients)], rotation=45, ha='right')
        ax.set_ylim(0, 1.05)
        ax.set_ylabel('within-client proportion')
        ax.set_title(f'{name}: P({col} | client)')
        if len(levels) <= 14:
            ax.legend(fontsize=7, loc='center left', bbox_to_anchor=(1.0, 0.5))
    plt.tight_layout()
    plt.show()


def sample_to_client(parts, key='filename'):
    m = {}
    for i, p in enumerate(parts):
        for k in p[key]:
            m[k] = i
    return m


def align_labels(reference_parts, other_parts, key='filename'):
    r = sample_to_client(reference_parts, key)
    o = sample_to_client(other_parts, key)
    keys = [k for k in r if k in o]
    return (np.array([r[k] for k in keys]),
            np.array([o[k] for k in keys]))

# Global partitioning config

`N_CLIENTS = 12` is the natural client count for P1 (6 designs × 2 quantile
bins of `clock_ns`). Every other strategy is aligned to the same count so
the JS-divergence and size-imbalance scores are directly comparable:

- P0 IID at 12 clients.
- P1 with `design_col='design_name'` × `n_persona_bins=2` = 6 × 2 = 12 clients.
- P2 at 12 clients across α ∈ {0.1, 0.5, 5.0}.
- P3 with `k = 12` clusters; ARI/NMI vs P1 tells us whether the natural
  metadata clusters agree with the hand-drawn (design, clock-tier) personas.

In [ ]:
N_CLIENTS = 12
SEED = 42
results = {}

# P0 — IID baseline

Stratifies on the flow attributes so each client gets a proportional slice of every
flow recipe. If FL degrades on P1/P2/P3 relative to P0, we know the non-IID
structure is what bit.

In [ ]:
p0 = IIDPartitioner(
    n_partitions=N_CLIENTS,
    mode='features',
    stratify_cols=['design_name','macro_count','macro_placement','power_mesh','filler_insertion'],
    seed=SEED,
).partition(df_meta)

results['P0_IID'] = summarize(p0, 'P0 IID')
plot_partition_composition(p0, 'P0 IID')

# P1 — Hierarchical deterministic (design_name × clock_bin)

6 designs × 2 quantile bins of `clock_ns` = 12 clients. Each client is one
(design_variant, performance-tier) team — this is the interpretable,
reproducible "main" strategy.

The bin edges are computed on `clock_ns` (2, 5, 20 ns → the low-clock bin is
the *high-frequency* team, and vice versa). The `_client_cell` column added
by the partitioner records which (design_name, clock-bin) each row belongs to.

In [ ]:
p1 = HierarchicalPersonaPartitioner(
    n_partitions=N_CLIENTS,
    design_col='design_name',
    persona_col='clock_ns',
    n_persona_bins=2,
    bin_method='quantile',
    seed=SEED,
).partition(df_meta)

print('P1 cells (design_name|clock_bin, sample count, clock values):')
for i, part in enumerate(p1):
    print(f'  c{i:2d}  {part["_client_cell"].iloc[0]:30s} n={len(part):5d}  '
          f'clock_ns={sorted(part["clock_ns"].unique())}')

results['P1_hier'] = summarize(p1, 'P1 Hierarchical (design_name x clock_bin)')
plot_partition_composition(p1, 'P1 Hierarchical',
                           cols=('design_name','clock_ns','utilization'))

# P2 — Feature Dirichlet on `design_name`

For each of the 6 designs, Dirichlet(α) proportions are drawn across the 12
clients and the design's samples split accordingly.

- α = 0.1 → each design lands almost entirely on one client (extreme non-IID,
  more skewed than P1 because the design-persona binding is randomized).
- α = 0.5 → moderate skew.
- α = 5.0 → near-IID quantitatively (design near-uniform across clients).

The α → non-IID curve below is the continuous heterogeneity knob you need
for the "performance vs degree of non-IID" plot in Phase 5.

In [ ]:
ALPHAS = [0.1, 0.5, 5.0]
p2_all = {}
for alpha in ALPHAS:
    p2 = FeatureDirichletPartitioner(
        n_partitions=N_CLIENTS,
        feature_cols=['design_name'],
        alpha=alpha,
        seed=SEED,
    ).partition(df_meta)
    p2_all[alpha] = p2
    results[f'P2_dir_alpha{alpha}'] = summarize(
        p2, f'P2 Feature-Dirichlet (design_name, alpha={alpha})'
    )

# Heterogeneity curve vs alpha
alpha_js = {c: [] for c in FACTOR_COLS}
for alpha in ALPHAS:
    for col in FACTOR_COLS:
        alpha_js[col].append(float(np.nanmean(
            js_divergence_to_global(p2_all[alpha], col, global_props[col])
        )))

fig, ax = plt.subplots(figsize=(10, 5))
for col in FACTOR_COLS:
    ax.plot(ALPHAS, alpha_js[col], marker='o', label=col)
ax.set_xscale('log')
ax.set_xlabel('Dirichlet alpha (log scale)')
ax.set_ylabel('Mean JS divergence to global marginal')
ax.set_title('P2 heterogeneity vs alpha  (feature_cols = [design_name])')
ax.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.show()

for a in ALPHAS:
    plot_partition_composition(p2_all[a], f'P2 alpha={a}',
                               cols=('design_name','clock_ns','utilization'))

# P3 — Data-driven k-prototypes (weighted-Gower proxy)

Encoding + k-means with the user-supplied weight schema (`design_name` in place
of `family`).

On N28:

- `size_class` is constant (`small`) → dropped automatically.
- `aspect_ratio` is constant (`1.0`) → dropped automatically.

So the effective P3 axes on N28 are `design_name` (w=2.0), `log_frequency` (1.5),
`utilization` (1.5), `power_mesh` (0.75), `macro_placement` (0.75), `filler`
(0.5). This is exactly the Phase-2 findings translated into a distance:
design_name dominates, persona knobs come next, flow variants get a light touch.

In [ ]:
FEATURE_SPECS = [
    ('design_name',      'nominal',  2.00),
    ('size_class',       'ordinal',  1.00),
    ('log_frequency',    'interval', 1.50),
    ('utilization',      'interval', 1.50),
    ('aspect_ratio',     'interval', 1.00),
    ('power_mesh',       'nominal',  0.75),
    ('macro_placement',  'nominal',  0.75),
    ('filler',           'nominal',  0.50),
]
ORDINAL_ORDERS = {'size_class': ['small', 'medium', 'large']}

p3_part = KPrototypesPartitioner(
    n_partitions=N_CLIENTS,
    feature_specs=FEATURE_SPECS,
    ordinal_orders=ORDINAL_ORDERS,
    seed=SEED,
    n_init=20,
)
p3 = p3_part.partition(df_meta)

print(f'Dropped features (missing/constant): {p3_part.dropped_features_}')
print(f'Encoded matrix shape: {p3_part.encoded_matrix_.shape}')
print(f'k-means inertia: {p3_part.kmeans_.inertia_:.2f}')

results['P3_kproto'] = summarize(p3, f'P3 K-prototypes (weighted, k={N_CLIENTS})')
plot_partition_composition(p3, 'P3 K-prototypes',
                           cols=('design_name','clock_ns','utilization'))

# P1 ↔ P3 alignment (persona validation)

If P3's data-driven clusters mostly match the P1 hand-drawn personas, the
personas are *validated* by the natural metadata structure. If they diverge,
we have a *discovery*: the metadata is grouping differently than we assumed.

Reported metrics:

- **ARI** (Adjusted Rand Index) — chance-corrected agreement, 0 = random,
  1 = identical partitions.
- **NMI** (Normalized Mutual Information) — information overlap, 0 = independent,
  1 = deterministic function of each other.
- **Confusion matrix** — P1 client vs P3 cluster; block-diagonal (after
  cluster relabeling) = agreement.

In [ ]:
labels_p1, labels_p3 = align_labels(p1, p3, key='filename')
ari = adjusted_rand_score(labels_p1, labels_p3)
nmi = normalized_mutual_info_score(labels_p1, labels_p3)
print(f'ARI(P1, P3) = {ari:.3f}')
print(f'NMI(P1, P3) = {nmi:.3f}')

cell_names = [p1[i]['_client_cell'].iloc[0] for i in range(len(p1))]
cm = pd.crosstab(
    pd.Series(labels_p1, name='P1 (design_name|clock_bin)').map(dict(enumerate(cell_names))),
    pd.Series(labels_p3, name='P3 (cluster)'),
)
print()
print(cm)

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(cm.values, cmap='Blues', aspect='auto')
ax.set_xlabel('P3 cluster id')
ax.set_ylabel('P1 client (design_name | clock_bin)')
ax.set_xticks(range(cm.shape[1]))
ax.set_xticklabels(cm.columns)
ax.set_yticks(range(cm.shape[0]))
ax.set_yticklabels(cm.index, fontsize=9)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        v = cm.values[i, j]
        if v > 0:
            ax.text(j, i, str(v), ha='center', va='center', fontsize=8,
                    color='white' if v > cm.values.max() / 2 else 'black')
ax.set_title(f'P1 vs P3 confusion  (ARI={ari:.3f}, NMI={nmi:.3f})')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

# Cross-strategy comparison

One-glance summary of all strategies at the same client count.
Look at the JS-divergence row for `design_name` (should be high for P1, and
monotone-decreasing in α for P2) and the JS row for `power_mesh` /
`macro_placement` / `filler_insertion` (should stay low for every strategy;
if it spikes, the flow-axis stratification isn't working).

In [ ]:
rows = []
for name, res in results.items():
    row = {
        'strategy':   name,
        'n_clients':  len(res['sizes']),
        'imbalance':  round(res['imbalance'], 2),
        'min_size':   min(res['sizes']),
        'max_size':   max(res['sizes']),
    }
    for col in FACTOR_COLS:
        row[f'JS[{col}]'] = round(res['js_mean'][col], 3)
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index('strategy')
print(summary_df.to_string())

primary_cols = [c for c in summary_df.columns if c.startswith('JS[') and
                any(k in c for k in ('design_name','clock_ns','utilization'))]
flow_cols    = [c for c in summary_df.columns if c.startswith('JS[') and
                any(k in c for k in ('macro_placement','power_mesh','filler_insertion'))]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, group, title in [(axes[0], primary_cols, 'JS on PRIMARY / PERSONA axes  '
                                                   '(should be HIGH for non-IID strategies)'),
                          (axes[1], flow_cols, 'JS on FLOW axes  '
                                                '(should stay LOW everywhere)')]:
    sub = summary_df[group]
    x = np.arange(len(sub.index))
    width = 0.8 / max(len(group), 1)
    for j, col in enumerate(group):
        ax.bar(x + j*width, sub[col].values, width=width, label=col.replace('JS[','').rstrip(']'))
    ax.set_xticks(x + width * (len(group) - 1) / 2)
    ax.set_xticklabels(sub.index, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('Mean JS divergence')
    ax.set_title(title)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Save partitions for Phase 5

One CSV per (strategy, client) written to `./partition_outputs/`. Each file lists
the `.npy` filenames that go to that client, ready to feed a Flower DataLoader.

In [ ]:
def save_partitions(parts, prefix):
    for i, p in enumerate(parts):
        p[['filename']].to_csv(
            os.path.join(OUT_DIR, f'{prefix}_client_{i:02d}.csv'), index=False
        )
    print(f'  saved {len(parts)} files: {prefix}_client_*.csv')

save_partitions(p0, 'P0_IID')
save_partitions(p1, 'P1_hier')
for alpha in ALPHAS:
    save_partitions(p2_all[alpha], f'P2_dir_alpha{alpha}')
save_partitions(p3, 'P3_kproto')
print(f'All partitions written to {OUT_DIR}/')